In [11]:
import os
import random
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [12]:
DATA_DIR = "archive/characters_train/characters_train"  
IMG_SIZE = 96

class SimpsonsDataset(Dataset):
    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

def build_samples(data_dir):
    classes = sorted([d for d in os.listdir(data_dir)
                      if os.path.isdir(os.path.join(data_dir, d))])
    class_to_idx = {c: i for i, c in enumerate(classes)}
    samples = []
    for c in classes:
        cdir = os.path.join(data_dir, c)
        for f in os.listdir(cdir):
            # skip macOS junk files
            if f.startswith("._"):
                continue
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(cdir, f), class_to_idx[c]))
    return samples, class_to_idx, classes

samples, class_to_idx, classes = build_samples(DATA_DIR)
print(f"Total images: {len(samples)} | Classes: {len(classes)}")
print("Classes:", classes)

# Save class mapping for inference
with open("class_to_idx.json", "w") as f:
    json.dump(class_to_idx, f, indent=2)

Total images: 16764 | Classes: 42
Classes: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler', 'comic_book_guy', 'disco_stu', 'edna_krabappel', 'fat_tony', 'gil', 'groundskeeper_willie', 'homer_simpson', 'kent_brockman', 'krusty_the_clown', 'lenny_leonard', 'lionel_hutz', 'lisa_simpson', 'maggie_simpson', 'marge_simpson', 'martin_prince', 'mayor_quimby', 'milhouse_van_houten', 'miss_hoover', 'moe_szyslak', 'ned_flanders', 'nelson_muntz', 'otto_mann', 'patty_bouvier', 'principal_skinner', 'professor_john_frink', 'rainier_wolfcastle', 'ralph_wiggum', 'selma_bouvier', 'sideshow_bob', 'sideshow_mel', 'snake_jailbird', 'troy_mcclure', 'waylon_smithers']


In [13]:
from collections import defaultdict, Counter
import math

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ---- STRATIFIED SPLIT (per-class 85/15) ----
by_class = defaultdict(list)
for path, lbl in samples:
    by_class[lbl].append((path, lbl))

train_samples, val_samples = [], []
rng = random.Random(SEED)
for lbl, items in by_class.items():
    rng.shuffle(items)
    n_val = max(1, int(0.15 * len(items)))  # at least 1 val sample per class
    val_samples.extend(items[:n_val])
    train_samples.extend(items[n_val:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)

train_ds = SimpsonsDataset(train_samples, class_to_idx, transform=train_tf)
val_ds   = SimpsonsDataset(val_samples,   class_to_idx, transform=val_tf)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# ---- MILD oversampling only (sqrt of inverse freq, NOT inverse freq) ----
# This gently helps minority classes without destroying majority class learning.
label_counts = Counter([lbl for _, lbl in train_samples])
print("Class distribution (train):")
for c, i in class_to_idx.items():
    print(f"  {c:30s} {label_counts[i]}")

# sqrt softens the imbalance correction — majority classes still seen often
sample_weights = [1.0 / math.sqrt(label_counts[lbl]) for _, lbl in train_samples]
sampler = torch.utils.data.WeightedRandomSampler(
    sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

Train: 14270 | Val: 2494
Class distribution (train):
  abraham_grampa_simpson         622
  agnes_skinner                  29
  apu_nahasapeemapetilon         425
  barney_gumble                  73
  bart_simpson                   913
  carl_carlson                   68
  charles_montgomery_burns       812
  chief_wiggum                   671
  cletus_spuckler                33
  comic_book_guy                 320
  disco_stu                      6
  edna_krabappel                 312
  fat_tony                       19
  gil                            19
  groundskeeper_willie           83
  homer_simpson                  1528
  kent_brockman                  340
  krusty_the_clown               821
  lenny_leonard                  211
  lionel_hutz                    2
  lisa_simpson                   922
  maggie_simpson                 88
  marge_simpson                  879
  martin_prince                  49
  mayor_quimby                   168
  milhouse_van_houten            7

In [14]:
# Bump image size for richer features. If too slow, drop to 96.
IMG_SIZE = 128  # override

# Rebuild transforms with new size (re-run loaders)
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
train_ds = SimpsonsDataset(train_samples, class_to_idx, transform=train_tf)
val_ds   = SimpsonsDataset(val_samples,   class_to_idx, transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

class SimpsonsCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1),
                nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1),
                nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 32),    # 128 -> 64
            block(32, 64),   # 64 -> 32
            block(64, 128),  # 32 -> 16
            block(128, 256), # 16 -> 8
            block(256, 256), # 8  -> 4
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x); x = self.gap(x)
        return self.classifier(x)

model = SimpsonsCNN(len(classes)).to(device)
print("Params:", sum(p.numel() for p in model.parameters()))

Params: 2393674


In [15]:
EPOCHS = 30
LR = 1e-3

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # Validation
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            _, pred = out.max(1)
            all_p.extend(pred.cpu().numpy())
            all_t.extend(labels.numpy())

    val_acc  = float(np.mean(np.array(all_p) == np.array(all_t)))
    macro_f1 = f1_score(all_t, all_p, average="macro", zero_division=0)
    scheduler.step()

    print(f"Epoch {epoch}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_acc={val_acc:.4f} macro_f1={macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save({
            "model_state": model.state_dict(),
            "class_to_idx": class_to_idx,
            "img_size": IMG_SIZE,
        }, "model.pth")
        print(f"  ✔ Saved best model (macro_f1={macro_f1:.4f})")

print(f"\nBest macro F1: {best_f1:.4f}")

Epoch 1/30: 100%|██████████| 223/223 [09:23<00:00,  2.53s/it]


Epoch 1: train_loss=3.1111 train_acc=0.1849 | val_acc=0.3733 macro_f1=0.1412
  ✔ Saved best model (macro_f1=0.1412)


Epoch 2/30: 100%|██████████| 223/223 [10:32<00:00,  2.84s/it]


Epoch 2: train_loss=2.3936 train_acc=0.3909 | val_acc=0.5814 macro_f1=0.2790
  ✔ Saved best model (macro_f1=0.2790)


Epoch 3/30: 100%|██████████| 223/223 [32:27<00:00,  8.73s/it]    


...

Epoch 28: train_loss=0.5785 train_acc=0.9715 | val_acc=0.9563 macro_f1=0.9103


Epoch 29/30: 100%|██████████| 223/223 [09:17<00:00,  2.50s/it]


Epoch 29: train_loss=0.5762 train_acc=0.9704 | val_acc=0.9563 macro_f1=0.9288
  ✔ Saved best model (macro_f1=0.9288)


Epoch 30/30: 100%|██████████| 223/223 [09:20<00:00,  2.51s/it]


Epoch 30: train_loss=0.5726 train_acc=0.9732 | val_acc=0.9571 macro_f1=0.9302
  ✔ Saved best model (macro_f1=0.9302)

Best macro F1: 0.9302


In [16]:
ckpt = torch.load("model.pth", map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

all_p, all_t = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        _, pred = out.max(1)
        all_p.extend(pred.cpu().numpy())
        all_t.extend(labels.numpy())

print(classification_report(all_t, all_p, target_names=classes, zero_division=0))
print("Final Macro F1:", f1_score(all_t, all_p, average="macro", zero_division=0))
print("Model saved at: model.pth")

                          precision    recall  f1-score   support

  abraham_grampa_simpson       0.99      0.95      0.97       109
           agnes_skinner       0.83      1.00      0.91         5
  apu_nahasapeemapetilon       1.00      1.00      1.00        74
           barney_gumble       0.85      0.92      0.88        12
            bart_simpson       0.93      0.94      0.94       161
            carl_carlson       1.00      1.00      1.00        11
charles_montgomery_burns       0.94      0.95      0.94       143
            chief_wiggum       0.96      0.96      0.96       118
         cletus_spuckler       0.80      0.80      0.80         5
          comic_book_guy       0.92      0.88      0.90        56
               disco_stu       1.00      1.00      1.00         1
          edna_krabappel       0.93      1.00      0.96        54
                fat_tony       1.00      0.67      0.80         3
                     gil       0.75      1.00      0.86         3
    groun